In [ ]:
import pandas as pd
import pickle
import numpy as np


In [ ]:
df = pickle.load(open('dataset_level6.pkl','rb'))

In [ ]:
df
# batting_team
# bowling team
# city
# current score
# ball left
# wickets left
# current rr
# last five


In [ ]:
df.isnull().sum()

In [ ]:
df[df['city'].isnull()]['venue'].value_counts()

In [ ]:
cities = np.where(df['city'].isnull(),df['venue'].str.split().apply(lambda x:x[0]),df['city'])

In [ ]:
df['city'] = cities

In [ ]:
df.isnull().sum()

In [ ]:
df.drop(columns=['venue'],inplace=True)

In [ ]:
df

In [ ]:
eligible_cities = df['city'].value_counts()[df['city'].value_counts() > 600].index.tolist()

In [ ]:
df = df[df['city'].isin(eligible_cities)]

In [ ]:
df

In [ ]:
df['current_score'] = df.groupby('match_id').cumsum()['runs']

In [ ]:
df

In [ ]:
df['over'] = df['ball'].apply(lambda x:str(x).split(".")[0])
df['ball_no'] = df['ball'].apply(lambda x:str(x).split(".")[1])
df

In [ ]:
df['balls_bowled'] = (df['over'].astype('int')*6) + df['ball_no'].astype('int')
df

In [ ]:
df['balls_left'] = 120 - df['balls_bowled']
df['balls_left'] = df['balls_left'].apply(lambda x:0 if x<0 else x)
df

In [ ]:
df['player_dismissed'] = df['player_dismissed'].apply(lambda x:0 if x=='0' else 1)
df['player_dismissed'] = df['player_dismissed'].astype('int')
df['player_dismissed'] = df.groupby('match_id').cumsum()['player_dismissed']
df['wickets_left'] = 10 - df['player_dismissed']

In [ ]:
df

In [ ]:
df['crr'] = (df['current_score']*6)/df['balls_bowled']

In [ ]:
df

In [ ]:
groups = df.groupby('match_id')

match_ids = df['match_id'].unique()
last_five = []
for id in match_ids:
    last_five.extend(groups.get_group(id).rolling(window=30).sum()['runs'].values.tolist())

In [ ]:
df['last_five'] = last_five

In [ ]:
df[['batting_team','bowling_team','city','current_score','balls_left','wickets_left','crr','last_five']]

In [ ]:
final_df = df.groupby('match_id').sum()['runs'].reset_index().merge(df,on='match_id')

In [ ]:
final_df=final_df[['batting_team','bowling_team','city','current_score','balls_left','wickets_left','crr','last_five','runs_x']]

In [ ]:
final_df

In [ ]:
final_df.dropna(inplace=True)

In [ ]:
final_df.isnull().sum()

In [ ]:
final_df = final_df.sample(final_df.shape[0])

In [ ]:
final_df.sample(2)

In [ ]:
X = final_df.drop(columns=['runs_x'])
y = final_df['runs_x']
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [ ]:
X_train

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score,mean_absolute_error

In [ ]:
trf = ColumnTransformer([
    ('trf',OneHotEncoder(sparse=False,drop='first'),['batting_team','bowling_team','city'])
]
,remainder='passthrough')

In [ ]:
pipe = Pipeline(steps=[
    ('step1',trf),
    ('step2',StandardScaler()),
    ('step3',XGBRegressor(n_estimators=1000,learning_rate=0.2,max_depth=12,random_state=1))
])

In [ ]:
pipe.fit(X_train,y_train)
y_pred = pipe.predict(X_test)
print(r2_score(y_test,y_pred))
print(mean_absolute_error(y_test,y_pred))

In [ ]:
pickle.dump(pipe,open('pipe.pkl','wb'))

In [ ]:
import xgboost
xgboost.__version__